In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [37]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)



In [38]:
dataset = generate_dataset()
dataset

[{'task': "Extract the AWS region from an S3 URI in the format 's3://bucket-name/key' and return it, defaulting to 'us-east-1' if not explicitly specified in the URI",
  'format': 'python',
  'solution_criteria': 'Function should parse S3 URIs, extract region information if present, and return the correct AWS region. Should handle standard S3 and S3-compatible URIs.'},
 {'task': "Create a JSON configuration object for an AWS Lambda function that specifies runtime as 'python3.11', timeout of 60 seconds, memory of 256 MB, and environment variables for DB_HOST and API_KEY",
  'format': 'json',
  'solution_criteria': 'JSON should be valid, include all specified properties with correct AWS Lambda configuration structure, and properly format environment variables.'},
 {'task': 'Create a regular expression that matches valid AWS IAM role ARN formats (arn:aws:iam::account-id:role/role-name), where account-id is a 12-digit number and role-name contains alphanumeric characters, hyphens, and unde

In [39]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [33]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    please solve the following task:
    {test_case["task"]}

    * Respond only with Python, JSON or a plain Regex
    * Do not add any comments or commentary or explanation
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [40]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [34]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [35]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (syntax_score + model_score)/2
    return{
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [28]:
from statistics import mean 

def run_eval(dataset):
    """Loads the dataset and calls run_tes_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
    return results

In [41]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)

Average Score: 8.0


In [42]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\nimport json\n\ndef extract_aws_region(s3_uri):\n    # Pattern to match S3 URI with optional region syntax\n    # s3://bucket-name/key or s3://bucket-name.region/key\n    pattern = r's3://([^/.]+)(?:\\.([a-z0-9\\-]+))?(?:/.*)?$'\n    \n    match = re.match(pattern, s3_uri)\n    if match:\n        region = match.group(2)\n        return region if region else 'us-east-1'\n    return 'us-east-1'\n\n# Test cases\nprint(extract_aws_region('s3://my-bucket/my-key'))\nprint(extract_aws_region('s3://my-bucket.eu-west-1/my-key'))\nprint(extract_aws_region('s3://bucket-name'))\n",
    "test_case": {
      "task": "Extract the AWS region from an S3 URI in the format 's3://bucket-name/key' and return it, defaulting to 'us-east-1' if not explicitly specified in the URI",
      "format": "python",
      "solution_criteria": "Function should parse S3 URIs, extract region information if present, and return the correct AWS region. Should handle standard S3 and S3-compati